# 2.2 Scaled Agent Memory: Vector vs Bi-Temporal Graph vs Full Context on All of LongMemEval and LoCoMo

*Formerly notebook 15.*

**Presentation map:** Slide 52 (memory tiers: episodic → semantic), Slide 53 (bi-temporal edges — a contradiction
*invalidates* the old edge instead of deleting it), Slide 54 (Graphiti-style `add_episode`), Slide 55 (Zep/Graphiti on
LongMemEval), Slide 56 (no memory design wins every question type).

Notebook A3 compared the memories on a stratified sample of 108 LongMemEval questions; several per-type differences
(temporal reasoning, single-session-assistant) were too close to call, and the full-context baseline ran on only 28
questions. This notebook removes those limits:

| Benchmark | Questions | History per question | Source |
|---|---|---|---|
| **LongMemEval-S** (cleaned) | **all 500** (6 types + 30 abstention) | ≈48 sessions, ≈115k tokens, each question its own haystack | Wu et al., ICLR 2025; `xiaowu0162/longmemeval-cleaned`, MIT |
| **LoCoMo** | **1,540** (all except the adversarial category) | 10 long conversations between two people, 19–32 sessions each | Maharana et al., ACL 2024; `snap-research/locomo`, CC BY-NC 4.0 |

**Three memories, one answer model and prompt** (notebook A3's design, unchanged):

| Memory | Index | What the answerer sees |
|---|---|---|
| (a) **Vector memory** | every round (user message + reply; LoCoMo: every turn) embedded with its date | top-10 rounds |
| (b) **Bi-temporal graph memory** | the LLM extracts facts per session; entity resolution; *state* facts invalidate older ones (`invalid_at`), never deleted; provenance on every edge | top-25 facts with validity dates + the source turns of the top-5 |
| (c) **Full context** | none | the whole history (fits in the model's 1M-token window) — on **every** question this time |

**Scoring:** LongMemEval's official type-specific judge prompts (copied from `src/evaluation/evaluate_qa.py`); for
LoCoMo a generous correct/wrong judge in the style used by Mem0 and Zep. *Deviation:* the judge is `gpt-6-luna`.

**Model:** `gpt-6-luna`, no reasoning, via the OpenAI Batch API (`smash_batch.py`). Retrieval results are cached as
JSONL in `data/benchmarks/longmemeval/cache_luna/` and `data/benchmarks/locomo/cache/` (vectors are not stored).

In [ ]:
# Notebooks live in examples/<task group>/; data/, smash_batch.py and the .venv-* environments stay in examples/.
# Every path below is relative to examples/, so run from there whether the notebook is opened in its own folder or not.
import os, sys
if not os.path.exists("smash_batch.py") and os.path.exists(os.path.join("..", "smash_batch.py")):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [1]:
import json, os, re, time, random, hashlib, math
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, Literal
import numpy as np
from pydantic import BaseModel, Field
from smash_batch import LLM, mode_from_env, wilson, paired_boot, EMBED_MODEL

MODE = mode_from_env()
TEST = MODE == "test"
LME, LCM = "data/benchmarks/longmemeval", "data/benchmarks/locomo"
llm_a = LLM(f"{LME}/cache_luna" + ("_test" if TEST else ""))
llm_b = LLM(f"{LCM}/cache" + ("_test" if TEST else ""))

def jsonl(path):
    return {r["key"]: r for r in map(json.loads, open(path))} if os.path.exists(path) else {}

def append(path, row):
    with open(path, "a") as f:
        f.write(json.dumps(row) + "\n")

def embed(texts, cap=6000):
    # Uncached embedding (the vectors are large; only retrieval results are cached). Returns (unit vectors, tokens).
    if MODE == "replay":
        raise RuntimeError("retrieval not cached - run with SMASH_MODE=test or batch")
    texts = [(t or " ")[:cap] for t in texts]
    out, used, i = [], 0, 0
    while i < len(texts):
        batch, size = [], 0
        while i < len(texts) and len(batch) < 512 and size + len(texts[i]) < 400_000:
            batch.append(texts[i]); size += len(texts[i]); i += 1
        for attempt in range(12):                     # rate limits: back off up to a minute per try
            try:
                r = llm_a.client.embeddings.create(model=EMBED_MODEL, input=batch); break
            except Exception:
                if attempt == 11:
                    raise
                time.sleep(min(60, 2 ** attempt) + random.random())
        out += [e.embedding for e in r.data]; used += r.usage.total_tokens
    v = np.array(out, np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True), used

def pmap(fn, items, workers=4):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(fn, items))

# Part A — LongMemEval-S, all 500 questions

In [2]:
data = json.load(open(f"{LME}/longmemeval_s_cleaned.json"))
by_id = {q["question_id"]: q for q in data}
is_abs = lambda q: q["question_id"].endswith("_abs")
TYPES = ["single-session-user", "single-session-assistant", "single-session-preference",
         "multi-session", "knowledge-update", "temporal-reasoning"]
group = lambda qid: "abstention" if is_abs(by_id[qid]) else by_id[qid]["question_type"]
QIDS = sorted(by_id)
if TEST:
    QIDS = [next(q for q in QIDS if group(q) == g) for g in TYPES + ["abstention"]]
print(f"LongMemEval-S: {len(QIDS)} questions |", dict(Counter(map(group, QIDS))))
n_sess = sum(len(by_id[q]["haystack_sessions"]) for q in QIDS)
print(f"{n_sess:,} sessions in the haystacks")

LongMemEval-S: 500 questions | {'single-session-user': 64, 'multi-session': 121, 'knowledge-update': 72, 'abstention': 30, 'single-session-preference': 30, 'temporal-reasoning': 127, 'single-session-assistant': 56}
23,867 sessions in the haystacks


## (a) Vector memory

In [3]:
ROUND_CAP, K_VEC = 1500, 10
VEC_PATH = f"{llm_a.cache}/retrieval_vector.jsonl"
vec_cache = jsonl(VEC_PATH)

def rounds_of(q):
    out = []
    for sid, date, sess in zip(q["haystack_session_ids"], q["haystack_dates"], q["haystack_sessions"]):
        for i in range(0, len(sess), 2):
            txt = "\n".join(f"{t['role'].upper()}: {t['content']}" for t in sess[i:i + 2])
            out.append({"sid": sid, "date": date, "text": f"[{date}] {txt}"})
    return out

def vector_retrieve(qid):
    if qid in vec_cache:
        return vec_cache[qid]
    q = by_id[qid]
    rs = rounds_of(q)
    E, used = embed([r["text"] for r in rs] + [q["question"]])
    top = sorted(np.argsort(-(E[:-1] @ E[-1]))[:K_VEC])
    row = {"key": qid, "context": "\n\n".join(rs[i]["text"][:ROUND_CAP] for i in top), "embed_tokens": used,
           "hit": any(rs[i]["sid"] in q["answer_session_ids"] for i in top)}
    append(VEC_PATH, row)
    return row

VEC = {r["key"]: r for r in pmap(vector_retrieve, QIDS)}
ans_q = [q for q in QIDS if not is_abs(by_id[q])]
print(f"Top-{K_VEC} rounds include an evidence session: {sum(VEC[q]['hit'] for q in ans_q)}/{len(ans_q)}")

Top-10 rounds include an evidence session: 465/470


## (b) Bi-temporal graph memory: one extraction request per unique session (one Batch job)

In [4]:
class Fact(BaseModel):
    subject: str = Field(description="entity the fact is about; use 'user' for the human user")
    relation: str = Field(description="UPPER_SNAKE_CASE relation, e.g. LIVES_IN, OWNS, PLANS_TO, RECOMMENDED")
    object: str = Field(description="value or entity; keep numbers, names and dates exact")
    fact: str = Field(description="one self-contained sentence with all specifics (names, numbers, dates)")
    turn: int = Field(description="index of the turn the fact comes from")
    event_date: Optional[str] = Field(description="YYYY/MM/DD if the fact has its own date (resolve 'last week' against the session date), else null")
    is_state: bool = Field(description="true if this value replaces earlier values of the same subject+relation (current job, address, count, status); false for events and multi-valued facts")

class Episode(BaseModel):
    facts: list[Fact]

EXTRACT_SYS = ("You build the long-term memory of a personal assistant. From one chat session, extract the facts "
               "worth remembering: everything the USER says about themselves (events, possessions, preferences, "
               "plans, people, places, numbers, dates, changes of state) and specific things the ASSISTANT "
               "provided (recommendations, names, numbers, lists the user may ask about later). Skip generic "
               "chit-chat. Be exhaustive for personal details; at most 40 facts.")
TURN_CAP, SESSION_CAP = 4000, 40000
skey = lambda sid, sess: hashlib.sha1((sid + json.dumps(sess)).encode()).hexdigest()

jobs = {}
for qid in QIDS:
    q = by_id[qid]
    for sid, date, sess in zip(q["haystack_session_ids"], q["haystack_dates"], q["haystack_sessions"]):
        jobs.setdefault(skey(sid, sess), (sid, date, sess))
reqs = {}
for k, (sid, date, sess) in jobs.items():
    turns = "\n".join(f"[turn {i}] {t['role'].upper()}: {t['content'][:TURN_CAP]}" for i, t in enumerate(sess))
    reqs[k] = LLM.body([{"role": "system", "content": EXTRACT_SYS},
                        {"role": "user", "content": f"Session date: {date}\n\n{turns[:SESSION_CAP]}"}],
                       max_tokens=6000, schema=Episode)
res = llm_a.run("extractions", reqs, parse=Episode)
EXT = {k: (v["parsed"] or {"facts": []})["facts"] for k, v in res.items()}
print(f"{len(EXT):,} unique sessions -> {sum(map(len, EXT.values())):,} facts "
      f"({sum(v['parsed'] is None for v in res.values())} sessions unparsable, kept without facts)")

19,195 unique sessions -> 122,597 facts (1 sessions unparsable, kept without facts)


In [5]:
SELF = {"i", "me", "my", "myself", "user", "the user", "you (user)", "human"}
def norm(name, self_names=SELF):
    n = re.sub(r"[^a-z0-9 ]+", " ", str(name).lower()).strip()
    n = re.sub(r"^(the|a|an|my) ", "", n)
    return "user" if self_names and (n in self_names or str(name).strip().lower() in self_names) else n

def resolve_and_invalidate(edges, emb, self_names=SELF):
    names = sorted({e["s"] for e in edges} | {norm(e["o"], self_names) for e in edges})
    canon = {n: n for n in names}
    if len(names) > 1:
        V = emb(names)
        for a in range(len(names)):
            for b in np.where((V[a] @ V[a + 1:].T) >= 0.92)[0] + a + 1:
                canon[names[b]] = canon[names[a]]
    for e in edges:
        e["s"] = canon[e["s"]]
    latest, inv = {}, 0
    for e in edges:                                  # edges are in ingestion (chronological) order
        if not e["state"]:
            continue
        k = (e["s"], e["r"]); old = latest.get(k)
        if (old is not None and old["sid"] != e["sid"]
                and canon.get(norm(old["o"], self_names), norm(old["o"], self_names)) != canon.get(norm(e["o"], self_names), norm(e["o"], self_names))):
            old["invalid_at"] = e["valid_at"]; inv += 1
        latest[k] = e
    return sum(1 for n in names if canon[n] != n), inv

def graph_context(edges, question, emb, K_FACTS=25, K_SRC=5, SRC_CAP=1200):
    V = emb([f"{e['s']} {e['r']} {e['o']}: {e['fact']}" for e in edges] + [question])
    sims = V[:-1] @ V[-1]
    qn = norm(question, None)
    sims += np.array([0.05 if len(e["s"]) > 3 and e["s"] != "user" and e["s"] in qn else 0.0 for e in edges])
    top = list(np.argsort(-sims)[:K_FACTS])
    lines = []
    for i in sorted(top, key=lambda i: edges[i]["valid_at"]):
        e = edges[i]
        status = f"superseded {e['invalid_at']}" if e["invalid_at"] else "current"
        lines.append(f"- [{e['valid_at']} | {status}] {e['fact']}  (session {e['sid']}, turn {e['turn']})")
    srcs = [f"[{edges[i]['created_at']} session {edges[i]['sid']} turn {edges[i]['turn']}] {edges[i]['src'][:SRC_CAP]}" for i in top[:K_SRC]]
    return "FACTS FROM MEMORY GRAPH:\n" + "\n".join(lines) + "\n\nSOURCE TURNS OF THE TOP FACTS:\n" + "\n\n".join(srcs), top

GR_PATH = f"{llm_a.cache}/retrieval_graph.jsonl"
gr_cache = jsonl(GR_PATH)
def graph_retrieve(qid):
    if qid in gr_cache:
        return gr_cache[qid]
    q = by_id[qid]
    order = sorted(range(len(q["haystack_dates"])), key=lambda i: q["haystack_dates"][i])
    edges = []
    for i in order:
        sid, date, sess = q["haystack_session_ids"][i], q["haystack_dates"][i], q["haystack_sessions"][i]
        for f in EXT[skey(sid, sess)]:
            t = f["turn"] if isinstance(f["turn"], int) and 0 <= f["turn"] < len(sess) else 0
            edges.append({"s": norm(f["subject"]), "r": f["relation"].upper(), "o": f["object"], "fact": f["fact"],
                          "sid": sid, "turn": t, "created_at": date[:10], "valid_at": f.get("event_date") or date[:10],
                          "invalid_at": None, "state": bool(f["is_state"]), "src": sess[t]["content"]})
    used = [0]
    def emb(texts):
        V, u = embed(texts, cap=2000); used[0] += u; return V
    merged, inv = resolve_and_invalidate(edges, emb)
    ctx, top = graph_context(edges, q["question"], emb)
    row = {"key": qid, "context": ctx, "n_edges": len(edges), "merged": merged, "invalidated": inv,
           "embed_tokens": used[0], "hit": any(edges[i]["sid"] in q["answer_session_ids"] for i in top)}
    append(GR_PATH, row)
    return row

GR = {r["key"]: r for r in pmap(graph_retrieve, QIDS)}
print(f"Graph memories: {np.mean([r['n_edges'] for r in GR.values()]):.0f} edges per history, "
      f"{sum(r['invalidated'] for r in GR.values()):,} edges invalidated (kept), "
      f"top-25 facts include an evidence session: {sum(GR[q]['hit'] for q in ans_q)}/{len(ans_q)}")

Graph memories: 304 edges per history, 2,534 edges invalidated (kept), top-25 facts include an evidence session: 446/470


## Answers (vector, graph and full context for every question) and the official judge

In [6]:
def anscheck_prompt(task, question, answer, response, abstention=False):
    # Copied from LongMemEval src/evaluation/evaluate_qa.py (get_anscheck_prompt).
    if not abstention:
        if task in ['single-session-user', 'single-session-assistant', 'multi-session']:
            template = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response is equivalent to the correct answer or contains all the intermediate steps to get the correct answer, you should also answer yes. If the response only contains a subset of the information required by the answer, answer no. \n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
        elif task == 'temporal-reasoning':
            template = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response is equivalent to the correct answer or contains all the intermediate steps to get the correct answer, you should also answer yes. If the response only contains a subset of the information required by the answer, answer no. In addition, do not penalize off-by-one errors for the number of days. If the question asks for the number of days/weeks/months, etc., and the model makes off-by-one errors (e.g., predicting 19 days when the answer is 18), the model's response is still correct. \n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
        elif task == 'knowledge-update':
            template = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response contains some previous information along with an updated answer, the response should be considered as correct as long as the updated answer is the required answer.\n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
        elif task == 'single-session-preference':
            template = "I will give you a question, a rubric for desired personalized response, and a response from a model. Please answer yes if the response satisfies the desired response. Otherwise, answer no. The model does not need to reflect all the points in the rubric. The response is correct as long as it recalls and utilizes the user's personal information correctly.\n\nQuestion: {}\n\nRubric: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
        else:
            raise NotImplementedError
    else:
        template = "I will give you an unanswerable question, an explanation, and a response from a model. Please answer yes if the model correctly identifies the question as unanswerable. The model could say that the information is incomplete, or some other information is given but the asked information is not.\n\nQuestion: {}\n\nExplanation: {}\n\nModel Response: {}\n\nDoes the model correctly identify the question as unanswerable? Answer yes or no only."
    return template.format(question, answer, response)

ANSWER_SYS = ("You are a personal assistant with a memory of your past conversations with the user. Use only the "
              "memory provided. Dates matter: compute elapsed time from the dates shown, and prefer the most recent "
              "value when a fact changed. If the memory does not contain the answer, say so. Think briefly, then "
              "give the final answer on the last line as 'Answer: ...'.")

def full_context(q):
    sess = sorted(zip(q["haystack_session_ids"], q["haystack_dates"], q["haystack_sessions"]), key=lambda x: x[1])
    return "\n\n".join(f"### Session {sid} ({d})\n" + "\n".join(f"{t['role'].upper()}: {t['content']}" for t in s)
                       for sid, d, s in sess)

METHODS = ["vector", "graph", "full"]
reqs = {}
for qid in QIDS:
    q = by_id[qid]
    for m in METHODS:
        ctx = VEC[qid]["context"] if m == "vector" else GR[qid]["context"] if m == "graph" else full_context(q)
        reqs[f"{m}|{qid}"] = LLM.body([{"role": "system", "content": ANSWER_SYS},
                                      {"role": "user", "content": f"Current date: {q['question_date']}\n\nMEMORY:\n{ctx}\n\nQuestion: {q['question']}"}],
                                     max_tokens=800)
ANS_A = llm_a.run("answers", reqs)
jud = llm_a.run("judgments", {k: LLM.body([{"role": "user", "content": anscheck_prompt(
    by_id[k.split("|")[1]]["question_type"], by_id[k.split("|")[1]]["question"], by_id[k.split("|")[1]]["answer"],
    a["text"], abstention=is_abs(by_id[k.split("|")[1]]))}], max_tokens=5) for k, a in ANS_A.items()})
OK_A = {k: "yes" in v["text"].lower() for k, v in jud.items()}
print(f"{len(ANS_A):,} answers, {len(OK_A):,} judgments")

1,500 answers, 1,500 judgments


In [7]:
def acc(ok, keys):
    k = sum(ok[x] for x in keys); lo, hi = wilson(k, len(keys))
    return k / len(keys), lo, hi

RES_A = {}
print(f"{'question group':<27}{'n':>4} | " + " | ".join(f"{m:^18}" for m in METHODS))
for g in TYPES + ["abstention", "ALL"]:
    qs = [q for q in QIDS if g == "ALL" or group(q) == g]
    RES_A[g] = {m: acc(OK_A, [f"{m}|{q}" for q in qs]) for m in METHODS}
    print(f"{g:<27}{len(qs):>4} | " + " | ".join(f"{a:5.0%} [{lo:.0%}-{hi:.0%}]".center(18) for a, lo, hi in RES_A[g].values()))
DIFF_A = {(a, b): paired_boot([OK_A[f"{a}|{q}"] for q in QIDS], [OK_A[f"{b}|{q}"] for q in QIDS])
          for a, b in [("vector", "graph"), ("full", "graph"), ("vector", "full")]}
for (a, b), (dm, lo, hi) in DIFF_A.items():
    print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]")
TOK_A = {m: np.mean([ANS_A[f"{m}|{q}"]["in"] for q in QIDS]) for m in METHODS}
print("Answer-prompt tokens per question: " + ", ".join(f"{m} {t:,.0f}" for m, t in TOK_A.items()))

question group                n |       vector       |       graph        |        full       
single-session-user          64 |    95% [87%-98%]   |    98% [92%-100%]  |    88% [77%-94%]  
single-session-assistant     56 |    91% [81%-96%]   |    52% [39%-64%]   |   100% [94%-100%] 
single-session-preference    30 |    73% [56%-86%]   |    70% [52%-83%]   |    47% [30%-64%]  
multi-session               121 |    60% [51%-68%]   |    76% [68%-83%]   |    57% [48%-65%]  
knowledge-update             72 |    88% [78%-93%]   |    94% [87%-98%]   |    88% [78%-93%]  
temporal-reasoning          127 |    61% [53%-69%]   |    73% [65%-80%]   |    53% [44%-61%]  
abstention                   30 |    97% [83%-99%]   |    90% [74%-97%]   |    97% [83%-99%]  
ALL                         500 |    75% [71%-79%]   |    79% [75%-82%]   |    71% [67%-75%]  
  graph - vector: +3.4% [-0.8%, +7.4%]
  graph - full: +7.8% [+3.0%, +12.6%]
  full - vector: -4.4% [-8.4%, -0.4%]
Answer-prompt tokens per quest

# Part B — LoCoMo: 10 long two-person conversations, 1,540 questions

In [8]:
locomo = json.load(open(f"{LCM}/locomo10.json"))
CAT = {1: "multi-hop", 2: "temporal", 3: "open-domain", 4: "single-hop"}
if TEST:
    locomo = locomo[:1]
convs = {}
for c in locomo:
    conv = c["conversation"]
    sessions = []
    for i in range(1, 100):
        if f"session_{i}" not in conv:
            break
        sessions.append({"sid": f"S{i}", "date": conv[f"session_{i}_date_time"], "turns": conv[f"session_{i}"]})
    qa = [dict(q, qid=f"{c['sample_id']}#{n}") for n, q in enumerate(c["qa"]) if q["category"] in CAT and "answer" in q]
    if TEST:
        qa = [q for cat in CAT for q in [x for x in qa if x["category"] == cat][:4]]
    convs[c["sample_id"]] = {"a": conv["speaker_a"], "b": conv["speaker_b"], "sessions": sessions, "qa": qa}
LQ = [(cid, q) for cid, cv in convs.items() for q in cv["qa"]]
print(f"LoCoMo: {len(convs)} conversations, {sum(len(c['sessions']) for c in convs.values())} sessions, "
      f"{len(LQ):,} questions |", dict(Counter(CAT[q['category']] for _, q in LQ)))

LoCoMo: 10 conversations, 272 sessions, 1,540 questions | {'temporal': 321, 'open-domain': 96, 'multi-hop': 282, 'single-hop': 841}


In [9]:
def turn_text(t, date):
    cap = f" [shares a photo: {t['blip_caption']}]" if t.get("blip_caption") else ""
    return f"[{date}] {t['speaker']}: {t['text']}{cap}"

LVEC_PATH = f"{llm_b.cache}/retrieval_vector.jsonl"
lvec_cache = jsonl(LVEC_PATH)
def lvector(item):
    cid, q = item
    if q["qid"] in lvec_cache:
        return lvec_cache[q["qid"]]
    cv = convs[cid]
    turns = [(s["sid"], t, turn_text(t, s["date"])) for s in cv["sessions"] for t in s["turns"]]
    E, used = embed([x[2] for x in turns] + [q["question"]])
    top = sorted(np.argsort(-(E[:-1] @ E[-1]))[:K_VEC])
    row = {"key": q["qid"], "context": "\n".join(turns[i][2] for i in top), "embed_tokens": used,
           "hit": any(turns[i][1]["dia_id"] in q.get("evidence", []) for i in top)}
    append(LVEC_PATH, row)
    return row
LVEC = {r["key"]: r for r in pmap(lvector, LQ)}

LEXTRACT_SYS = ("You build the long-term memory of an assistant that will answer questions about two people. From one "
                "conversation session between them, extract every fact worth remembering about each person: events "
                "(with dates), possessions, preferences, plans, family, friends, places, numbers, feelings and changes "
                "of state. Use the person's name as subject. Resolve relative dates ('last week', 'yesterday') against "
                "the session date. At most 40 facts.")
reqs = {}
for cid, cv in convs.items():
    for s in cv["sessions"]:
        turns = "\n".join(f"[turn {i}] {t['speaker']}: {t['text']}" + (f" [shares a photo: {t['blip_caption']}]" if t.get("blip_caption") else "")
                          for i, t in enumerate(s["turns"]))
        reqs[f"{cid}|{s['sid']}"] = LLM.body([{"role": "system", "content": LEXTRACT_SYS},
                                              {"role": "user", "content": f"Session date: {s['date']}\nSpeakers: {cv['a']} and {cv['b']}\n\n{turns}"}],
                                             max_tokens=6000, schema=Episode)
lres = llm_b.run("extractions", reqs, parse=Episode)
LEXT = {k: (v["parsed"] or {"facts": []})["facts"] for k, v in lres.items()}
print(f"LoCoMo extraction: {len(LEXT)} sessions -> {sum(map(len, LEXT.values())):,} facts")

from datetime import datetime
def parse_date(s):
    try:
        return datetime.strptime(s.strip(), "%I:%M %p on %d %B, %Y").strftime("%Y/%m/%d")
    except Exception:
        return s

LGR_PATH = f"{llm_b.cache}/retrieval_graph.jsonl"
lgr_cache = jsonl(LGR_PATH)
def lgraph(item):
    cid, q = item
    if q["qid"] in lgr_cache:
        return lgr_cache[q["qid"]]
    cv = convs[cid]
    edges = []
    for s in cv["sessions"]:
        d = parse_date(s["date"])
        for f in LEXT[f"{cid}|{s['sid']}"]:
            t = f["turn"] if isinstance(f["turn"], int) and 0 <= f["turn"] < len(s["turns"]) else 0
            edges.append({"s": norm(f["subject"], None), "r": f["relation"].upper(), "o": f["object"], "fact": f["fact"],
                          "sid": s["sid"], "turn": t, "created_at": d, "valid_at": f.get("event_date") or d,
                          "invalid_at": None, "state": bool(f["is_state"]),
                          "src": f"{s['turns'][t]['speaker']}: {s['turns'][t]['text']}", "dia": s["turns"][t]["dia_id"]})
    used = [0]
    def emb(texts):
        V, u = embed(texts, cap=2000); used[0] += u; return V
    merged, inv = resolve_and_invalidate(edges, emb, self_names=None)
    ctx, top = graph_context(edges, q["question"], emb)
    row = {"key": q["qid"], "context": ctx, "embed_tokens": used[0], "invalidated": inv,
           "hit": any(edges[i]["dia"] in q.get("evidence", []) for i in top)}
    append(LGR_PATH, row)
    return row
LGR = {r["key"]: r for r in pmap(lgraph, LQ)}

def lfull(cv):
    return "\n\n".join(f"### Session {s['sid']} ({s['date']})\n" + "\n".join(turn_text(t, s["date"]).split("] ", 1)[1] for t in s["turns"])
                       for s in cv["sessions"])

LANSWER_SYS = ("You answer questions about two people using only the memory of their past conversations provided. "
               "Dates matter: resolve relative dates against the session dates. Answer concisely; if the memory does not "
               "contain the answer, say so. Give the final answer on the last line as 'Answer: ...'.")
reqs = {}
for cid, q in LQ:
    for m in METHODS:
        ctx = LVEC[q["qid"]]["context"] if m == "vector" else LGR[q["qid"]]["context"] if m == "graph" else lfull(convs[cid])
        reqs[f"{m}|{q['qid']}"] = LLM.body([{"role": "system", "content": LANSWER_SYS},
                                           {"role": "user", "content": f"MEMORY:\n{ctx}\n\nQuestion: {q['question']}"}], max_tokens=400)
ANS_B = llm_b.run("answers", reqs)

class Judgment(BaseModel):
    label: Literal["CORRECT", "WRONG"]

QB = {q["qid"]: q for _, q in LQ}
def ljudge(k, response):
    q = QB[k.split("|", 1)[1]]
    return ("You are grading an answer to a question about a past conversation. Label it CORRECT if it contains the "
            "gold answer or something with the same meaning (be generous with wording; for dates, the same day, month "
            "or year as the gold answer counts, even in another format). Label it WRONG otherwise.\n\n"
            f"Question: {q['question']}\nGold answer: {q['answer']}\nResponse: {response}")
lj = llm_b.run("judgments", {k: LLM.body([{"role": "user", "content": ljudge(k, a["text"])}], max_tokens=20, schema=Judgment)
                             for k, a in ANS_B.items()}, parse=Judgment)
OK_B = {k: (v["parsed"] or {"label": "WRONG"})["label"] == "CORRECT" for k, v in lj.items()}

RES_B = {}
print(f"{'category':<14}{'n':>5} | " + " | ".join(f"{m:^18}" for m in METHODS))
for g in list(CAT.values()) + ["ALL"]:
    qs = [q["qid"] for _, q in LQ if g == "ALL" or CAT[q["category"]] == g]
    RES_B[g] = {m: acc(OK_B, [f"{m}|{x}" for x in qs]) for m in METHODS}
    print(f"{g:<14}{len(qs):>5} | " + " | ".join(f"{a:5.0%} [{lo:.0%}-{hi:.0%}]".center(18) for a, lo, hi in RES_B[g].values()))
lq = [q["qid"] for _, q in LQ]
DIFF_B = {(a, b): paired_boot([OK_B[f"{a}|{x}"] for x in lq], [OK_B[f"{b}|{x}"] for x in lq])
          for a, b in [("vector", "graph"), ("full", "graph"), ("vector", "full")]}
for (a, b), (dm, lo, hi) in DIFF_B.items():
    print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]")
TOK_B = {m: np.mean([ANS_B[f"{m}|{x}"]["in"] for x in lq]) for m in METHODS}
print("Answer-prompt tokens per question: " + ", ".join(f"{m} {t:,.0f}" for m, t in TOK_B.items()))

LoCoMo extraction: 272 sessions -> 3,617 facts


category          n |       vector       |       graph        |        full       
multi-hop       282 |    42% [36%-48%]   |    55% [49%-60%]   |    61% [56%-67%]  
temporal        321 |    69% [64%-74%]   |    79% [75%-83%]   |    75% [70%-80%]  
open-domain      96 |    42% [32%-52%]   |    36% [28%-46%]   |    43% [33%-53%]  
single-hop      841 |    77% [74%-80%]   |    81% [79%-84%]   |    90% [88%-92%]  
ALL            1540 |    67% [64%-69%]   |    73% [71%-75%]   |    79% [77%-81%]  
  graph - vector: +6.6% [+4.2%, +9.0%]
  graph - full: -5.5% [-7.6%, -3.4%]
  full - vector: +12.0% [+9.7%, +14.4%]
Answer-prompt tokens per question: vector 611, graph 1,393, full 20,721


## Cost

In [10]:
print("LongMemEval:"); LA = llm_a.print_ledger()
print("\nLoCoMo:"); LB = llm_b.print_ledger()
emb_tokens = sum(r["embed_tokens"] for r in list(VEC.values()) + list(GR.values()) + list(LVEC.values()) + list(LGR.values()))
print(f"\nRetrieval embeddings (not in the ledgers above): {emb_tokens / 1e6:.1f}M tokens, ${emb_tokens * 0.02 / 1e6:.2f}")

LongMemEval:
stage                           calls     input tok   output tok      USD  via Batch
answers                         1,500    54,813,113       73,666     2.76       100%
extractions                    19,195    50,170,160    9,393,831     4.86       100%
judgments                       1,500       291,252        6,000     0.02       100%
nb15_mask_ans_full                500    52,054,489       25,517     2.61       100%
nb15_mask_ans_graph               500       945,515       22,716     0.05       100%
nb15_mask_judge_full              500        98,047        2,000     0.01       100%
nb15_mask_judge_graph             500        95,270        2,000     0.01       100%
nb15_sess_ans_lme                 500     1,546,970       23,874     0.08       100%
nb15_sess_judge_lme               500        96,410        2,000     0.01       100%
nb15_sess_rerank_lme              500     2,219,112       25,027     0.12       100%
(embeddings)                        0             0 

In [11]:
print("=" * 96)
print(f"RESULTS SUMMARY — Notebook 2.2: agent memory, gpt-6-luna (no reasoning), {MODE} run")
print("=" * 96)
for name, RES, DIFF, TOK, n in (("LongMemEval-S", RES_A, DIFF_A, TOK_A, len(QIDS)), ("LoCoMo", RES_B, DIFF_B, TOK_B, len(LQ))):
    print(f"\n{name} (n = {n}): " + " | ".join(f"{m} {RES['ALL'][m][0]:.1%} [{RES['ALL'][m][1]:.0%}-{RES['ALL'][m][2]:.0%}]" for m in METHODS))
    for (a, b), (dm, lo, hi) in DIFF.items():
        print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]" + ("  resolvable" if lo > 0 or hi < 0 else "  not resolvable"))
    print("  answer-prompt tokens per question: " + ", ".join(f"{m} {t:,.0f}" for m, t in TOK.items()))
    best = {g: max(METHODS, key=lambda m: RES[g][m][0]) for g in RES if g != "ALL"}
    print("  best memory per question type: " + "; ".join(f"{g}: {m}" for g, m in best.items()))
print("\nLecture message (Slides 52-56): compare the per-type rows — the best memory changes with the question type,"
      "\nand full context is now affordable as a baseline, so a memory layer must beat it on accuracy or on tokens.")

RESULTS SUMMARY — Notebook 2.2: agent memory, gpt-6-luna (no reasoning), replay run

LongMemEval-S (n = 500): vector 75.2% [71%-79%] | graph 78.6% [75%-82%] | full 70.8% [67%-75%]
  graph - vector: +3.4% [-0.8%, +7.4%]  not resolvable
  graph - full: +7.8% [+3.0%, +12.6%]  resolvable
  full - vector: -4.4% [-8.4%, -0.4%]  resolvable
  answer-prompt tokens per question: vector 3,197, graph 2,063, full 104,365
  best memory per question type: single-session-user: graph; single-session-assistant: full; single-session-preference: vector; multi-session: graph; knowledge-update: graph; temporal-reasoning: graph; abstention: vector

LoCoMo (n = 1540): vector 66.8% [64%-69%] | graph 73.3% [71%-75%] | full 78.8% [77%-81%]
  graph - vector: +6.6% [+4.2%, +9.0%]  resolvable
  graph - full: -5.5% [-7.6%, -3.4%]  resolvable
  full - vector: +12.0% [+9.7%, +14.4%]  resolvable
  answer-prompt tokens per question: vector 611, graph 1,393, full 20,721
  best memory per question type: multi-hop: full; t

# Amendment (2026-11-08): session-level vector memory with rerank

*Added 2026-11-08, after the confidence run, as a sensitivity analysis (`baselines.md`, section 6, item 9). The
original primary result above stands and is not replaced. This protocol is written down before the run and was not
tuned.*

**Why.** Memory (a) retrieves single rounds (top 10) and scores 75.2% on LongMemEval-S, below published tuned vector
memories (Emergence 82.4%, Mastra about 80%), which retrieve whole sessions and rerank them. We run **one** variant of
that kind, fixed in advance, and run it once.

**(d) Session memory: the fixed variant**
1. **Index.** Each haystack session (LongMemEval) or conversation session (LoCoMo) is embedded as a whole: its date and
   all turns, with `text-embedding-3-small` and the same uncached `embed` helper as (a). If a session is longer than
   12,000 characters (a conservative bound under the model's 8,191-token input limit), it is split into consecutive
   12,000-character chunks and the session's similarity is the **maximum** over its chunks.
2. **Retrieve** the top **20** sessions by cosine similarity to the question (LoCoMo conversations have 19–32
   sessions, so 20 is most of a conversation).
3. **LLM rerank** (one call per question): the model sees the question (and, for LongMemEval, the question date) and,
   for each of the 20 sessions, its date and its first 150 words, and returns the session numbers in order of
   usefulness. Numbers it leaves out, or a reply that does not parse, fall back to embedding order.
4. **Keep the top sessions in full** in rerank order up to a **fixed token budget equal to what (a) passes to the
   answer step**: the mean size of (a)'s retrieved context, measured from 15's ledger (answer-prompt tokens regressed
   on context characters). The cell below measures it: **≈2,748 tokens on LongMemEval, ≈423 tokens on LoCoMo**. The
   first session that does not fit is cut to fill the remaining budget, and packing stops. The kept sessions are
   shown in chronological order, with their dates and without session ids.
5. **Answer and judge** with 15's own answer prompts (`ANSWER_SYS` / `LANSWER_SYS`, same user-message format and
   `max_tokens`) and 15's own judges (LongMemEval's official type-specific prompts; the LoCoMo CORRECT/WRONG judge).

**Questions.** All of them: 500 LongMemEval-S questions and 1,540 LoCoMo questions.

**Metric and contrasts.** Judge accuracy (Wilson 95% intervals), and per benchmark three paired-bootstrap differences
(`paired_boot`, 5,000 resamples, seed 7): **graph − session memory**, **session memory − round-level vector memory
(a)**, and **full context − session memory**. A difference is resolvable when its 95% interval excludes 0.

**Reading.** The verdict line names both results: the original (graph vs round-level vector memory) and this one
(graph vs session memory). If this contrast is used in the paper's forest plot it counts in its correction for
multiple comparisons. The graph memory gets no matching tuning here; a reranker for the graph's facts would be a
separate amendment.

New LLM stages (new cache entries only): `nb15_sess_rerank_lme`, `nb15_sess_ans_lme`, `nb15_sess_judge_lme`,
`nb15_sess_rerank_locomo`, `nb15_sess_ans_locomo`, `nb15_sess_judge_locomo`; retrieval results are cached in
`retrieval_session.jsonl` next to the existing retrieval caches.

In [12]:
# (d) Session memory - shared pieces: the token budget measured from (a), the chunked session embedding, the rerank.
K_SESS, SESS_CHUNK, PREVIEW_WORDS = 20, 12_000, 150

def vector_budget(ANS, VECS, keys):
    # Answer-prompt tokens of (a) regressed on its context characters: tokens per character b, and the budget
    # = b x mean context characters (the tokens (a) passes to the answer step). Packing is done in characters.
    tin = np.array([ANS[f"vector|{k}"]["in"] for k in keys], float)
    ch = np.array([len(VECS[k]["context"]) for k in keys], float)
    a, b = np.linalg.lstsq(np.vstack([np.ones_like(ch), ch]).T, tin, rcond=None)[0]
    return b * ch.mean(), int(round(ch.mean())), b

class Rerank(BaseModel):
    order: list[int] = Field(description="session numbers, most useful first; include every session")

RERANK_SYS = ("You rank past conversation sessions by how likely they are to contain the information needed to answer "
              "a question. For each session you see only its date and its opening words. Return all session numbers, "
              "most useful first.")

def rerank_body(question, previews, qdate=None):
    head = (f"Question date: {qdate}\n" if qdate else "") + f"Question: {question}\n\nSESSIONS:\n"
    body = "\n\n".join(f"[{n}] {d}: {p}" for n, (d, p) in enumerate(previews, 1))
    return LLM.body([{"role": "system", "content": RERANK_SYS}, {"role": "user", "content": head + body}],
                    max_tokens=300, schema=Rerank)

def rerank_order(res, n):
    # Positions 0..n-1 in rerank order; anything missing or unparsable falls back to embedding order.
    got = [] if res is None or res.get("parsed") is None else res["parsed"]["order"]
    seen, out = set(), []
    for x in got:
        if isinstance(x, int) and 1 <= x <= n and x not in seen:
            seen.add(x); out.append(x - 1)
    return out + [i for i in range(n) if i + 1 not in seen], len(seen)

def pack(blocks, order, budget_chars):
    # blocks[i] = (sort key, text). Take sessions in rerank order while they fit; cut the first one that does not.
    kept, used = [], 0
    for i in order:
        key, txt = blocks[i]
        room = budget_chars - used
        if room <= 0:
            break
        if len(txt) <= room:
            kept.append((key, i, txt)); used += len(txt) + 2
        else:
            kept.append((key, i, txt[:max(0, room - 6)] + " [...]")); break
    kept.sort()
    return "\n\n".join(t for _, _, t in kept), [i for _, i, _ in kept]

def chunks_of(text):
    return [text[j:j + SESS_CHUNK] for j in range(0, len(text), SESS_CHUNK)] or [" "]

_SESS_VEC = {}                                  # session -> chunk vectors, shared across questions within this run
def session_sims(keys, texts, question):
    # Embed new sessions (chunked), the question; similarity per session = max over its chunks.
    new = {k: t for k, t in zip(keys, texts) if k not in _SESS_VEC}
    flat = [(k, c) for k, t in new.items() for c in chunks_of(t)]
    E, used = embed([c for _, c in flat] + [question], cap=SESS_CHUNK)
    local = {}
    for (k, _), v in zip(flat, E[:-1]):
        local.setdefault(k, []).append(v)
    _SESS_VEC.update(local)                     # whole sessions only, so other threads never see a partial one
    qv = E[-1]
    return np.array([max(float(v @ qv) for v in local.get(k) or _SESS_VEC[k]) for k in keys]), used

# ---------------- Part A: LongMemEval-S ----------------
BUD_A_TOK, BUD_A, TPC_A = vector_budget(ANS_A, VEC, QIDS)
print(f"LongMemEval budget = (a)'s context: {BUD_A_TOK:,.0f} tokens ({BUD_A:,} characters, {1 / TPC_A:.2f} chars/token)")

def lme_sessions(q):
    out = []
    for sid, date, sess in zip(q["haystack_session_ids"], q["haystack_dates"], q["haystack_sessions"]):
        turns = "\n".join(f"{t['role'].upper()}: {t['content']}" for t in sess)
        out.append({"k": (skey(sid, sess), date), "sid": sid, "date": date, "turns": turns,
                    "block": f"### Session ({date})\n{turns}"})
    return out

SESS_PATH = f"{llm_a.cache}/retrieval_session.jsonl"
sess_cache = jsonl(SESS_PATH)
def session_retrieve(qid):
    if qid in sess_cache:
        return sess_cache[qid]
    q = by_id[qid]
    ss = lme_sessions(q)
    sims, used = session_sims([s["k"] for s in ss], [f"[{s['date']}]\n{s['turns']}" for s in ss], q["question"])
    top = [int(i) for i in np.argsort(-sims)[:K_SESS]]
    row = {"key": qid, "top": top, "sims": [round(float(sims[i]), 4) for i in top], "embed_tokens": used,
           "hit": any(ss[i]["sid"] in q["answer_session_ids"] for i in top)}
    append(SESS_PATH, row)
    return row

SR_A = {r["key"]: r for r in pmap(session_retrieve, QIDS)}
print(f"Top-{K_SESS} sessions include an evidence session: {sum(SR_A[q]['hit'] for q in ans_q)}/{len(ans_q)}")

reqs = {}
for qid in QIDS:
    q, ss = by_id[qid], lme_sessions(by_id[qid])
    prev = [(ss[i]["date"], " ".join(ss[i]["turns"].split()[:PREVIEW_WORDS])) for i in SR_A[qid]["top"]]
    reqs[qid] = rerank_body(q["question"], prev, q["question_date"])
RR_A = llm_a.run("nb15_sess_rerank_lme", reqs, parse=Rerank)

SCTX_A, SKEPT_A, n_ranked, n_same = {}, {}, [], 0
for qid in QIDS:
    q, ss, top = by_id[qid], lme_sessions(by_id[qid]), SR_A[qid]["top"]
    order, n_ok = rerank_order(RR_A.get(qid), len(top)); n_ranked.append(n_ok); n_same += order == list(range(len(top)))
    blocks = [(ss[i]["date"], ss[i]["block"]) for i in top]
    SCTX_A[qid], kept = pack(blocks, order, BUD_A)
    SKEPT_A[qid] = [ss[top[i]]["sid"] for i in kept]
print(f"Rerank: {np.mean(n_ranked):.1f} of {K_SESS} sessions ranked by the model on average, "
      f"{n_same} reranks identical to embedding order; "
      f"{np.mean([len(v) for v in SKEPT_A.values()]):.2f} sessions kept per question; kept sessions include an evidence "
      f"session: {sum(any(s in by_id[x]['answer_session_ids'] for s in SKEPT_A[x]) for x in ans_q)}/{len(ans_q)}")

reqs = {f"session|{qid}": LLM.body([{"role": "system", "content": ANSWER_SYS},
                                    {"role": "user", "content": f"Current date: {by_id[qid]['question_date']}\n\nMEMORY:\n{SCTX_A[qid]}\n\nQuestion: {by_id[qid]['question']}"}],
                                   max_tokens=800) for qid in QIDS}
SANS_A = llm_a.run("nb15_sess_ans_lme", reqs)
sj = llm_a.run("nb15_sess_judge_lme", {k: LLM.body([{"role": "user", "content": anscheck_prompt(
    by_id[k.split("|")[1]]["question_type"], by_id[k.split("|")[1]]["question"], by_id[k.split("|")[1]]["answer"],
    a["text"], abstention=is_abs(by_id[k.split("|")[1]]))}], max_tokens=5) for k, a in SANS_A.items()})
OK_A2 = dict(OK_A, **{k: "yes" in v["text"].lower() for k, v in sj.items()})
print(f"{len(SANS_A):,} session-memory answers; answer-prompt tokens per question: session "
      f"{np.mean([SANS_A[f'session|{q}']['in'] for q in QIDS]):,.0f} vs vector {TOK_A['vector']:,.0f}")

LongMemEval budget = (a)'s context: 2,748 tokens (13,862 characters, 5.04 chars/token)
Top-20 sessions include an evidence session: 470/470


Rerank: 19.5 of 20 sessions ranked by the model on average, 79 reranks identical to embedding order; 1.52 sessions kept per question; kept sessions include an evidence session: 439/470
500 session-memory answers; answer-prompt tokens per question: session 3,094 vs vector 3,197


In [13]:
# ---------------- Part B: LoCoMo ----------------
BUD_B_TOK, BUD_B, TPC_B = vector_budget(ANS_B, LVEC, lq)
print(f"LoCoMo budget = (a)'s context: {BUD_B_TOK:,.0f} tokens ({BUD_B:,} characters, {1 / TPC_B:.2f} chars/token)")

def lcm_sessions(cv):
    out = []
    for n, s in enumerate(cv["sessions"]):
        turns = "\n".join(turn_text(t, s["date"]).split("] ", 1)[1] for t in s["turns"])
        out.append({"sid": s["sid"], "date": s["date"], "turns": turns, "block": f"### Session ({s['date']})\n{turns}",
                    "n": n})
    return out
LSESS = {cid: lcm_sessions(cv) for cid, cv in convs.items()}

LSESS_PATH = f"{llm_b.cache}/retrieval_session.jsonl"
lsess_cache = jsonl(LSESS_PATH)
def lsession_conv(cid):
    # One embedding call per conversation: its sessions (chunked if long) + its uncached questions.
    todo = [q for q in convs[cid]["qa"] if q["qid"] not in lsess_cache]
    if not todo:
        return
    ss = LSESS[cid]
    flat = [(i, c) for i, s in enumerate(ss) for c in chunks_of(f"[{s['date']}]\n{s['turns']}")]
    E, used = embed([c for _, c in flat] + [q["question"] for q in todo], cap=SESS_CHUNK)
    C, Q = E[:len(flat)], E[len(flat):]
    owner = np.array([i for i, _ in flat])
    for j, q in enumerate(todo):
        cs = C @ Q[j]
        sims = np.array([cs[owner == i].max() for i in range(len(ss))])
        top = [int(i) for i in np.argsort(-sims)[:K_SESS]]
        ev = {e.split(":")[0].replace("D", "S") for e in q.get("evidence", [])}
        row = {"key": q["qid"], "top": top, "sims": [round(float(sims[i]), 4) for i in top],
               "embed_tokens": used if j == 0 else 0, "hit": any(ss[i]["sid"] in ev for i in top)}
        append(LSESS_PATH, row); lsess_cache[q["qid"]] = row
pmap(lsession_conv, list(convs))
SR_B = {x: lsess_cache[x] for x in lq}
print(f"Top-{K_SESS} sessions include an evidence session: {sum(SR_B[x]['hit'] for x in lq)}/{len(lq)}")

reqs = {}
for cid, q in LQ:
    ss = LSESS[cid]
    reqs[q["qid"]] = rerank_body(q["question"], [(ss[i]["date"], " ".join(ss[i]["turns"].split()[:PREVIEW_WORDS]))
                                                 for i in SR_B[q["qid"]]["top"]])
RR_B = llm_b.run("nb15_sess_rerank_locomo", reqs, parse=Rerank)

SCTX_B, SKEPT_B, n_ranked, n_same = {}, {}, [], 0
for cid, q in LQ:
    ss, top = LSESS[cid], SR_B[q["qid"]]["top"]
    order, n_ok = rerank_order(RR_B.get(q["qid"]), len(top)); n_ranked.append(n_ok); n_same += order == list(range(len(top)))
    SCTX_B[q["qid"]], kept = pack([(ss[i]["n"], ss[i]["block"]) for i in top], order, BUD_B)
    SKEPT_B[q["qid"]] = [ss[top[i]]["sid"] for i in kept]
ev_sess = lambda q: {e.split(":")[0].replace("D", "S") for e in q.get("evidence", [])}
print(f"Rerank: {np.mean(n_ranked):.1f} of up to {K_SESS} sessions ranked by the model on average, "
      f"{n_same} reranks identical to embedding order; "
      f"{np.mean([len(v) for v in SKEPT_B.values()]):.2f} sessions kept per question (the last one usually cut); kept "
      f"sessions include an evidence session: {sum(bool(set(SKEPT_B[x]) & ev_sess(QB[x])) for x in lq)}/{len(lq)}")

SANS_B = llm_b.run("nb15_sess_ans_locomo", {f"session|{q['qid']}": LLM.body(
    [{"role": "system", "content": LANSWER_SYS},
     {"role": "user", "content": f"MEMORY:\n{SCTX_B[q['qid']]}\n\nQuestion: {q['question']}"}], max_tokens=400)
    for _, q in LQ})
slj = llm_b.run("nb15_sess_judge_locomo", {k: LLM.body([{"role": "user", "content": ljudge(k, a["text"])}],
                                                       max_tokens=20, schema=Judgment) for k, a in SANS_B.items()},
                parse=Judgment)
OK_B2 = dict(OK_B, **{k: (v["parsed"] or {"label": "WRONG"})["label"] == "CORRECT" for k, v in slj.items()})
print(f"{len(SANS_B):,} session-memory answers; answer-prompt tokens per question: session "
      f"{np.mean([SANS_B[f'session|{x}']['in'] for x in lq]):,.0f} vs vector {TOK_B['vector']:,.0f}")

LoCoMo budget = (a)'s context: 423 tokens (2,015 characters, 4.77 chars/token)
Top-20 sessions include an evidence session: 1475/1540


Rerank: 19.2 of up to 20 sessions ranked by the model on average, 86 reranks identical to embedding order; 1.07 sessions kept per question (the last one usually cut); kept sessions include an evidence session: 737/1540
1,540 session-memory answers; answer-prompt tokens per question: session 552 vs vector 611


In [14]:
# Amendment results: accuracy per question type and the three pre-registered contrasts per benchmark.
M4 = METHODS + ["session"]
CONTRASTS = [("session", "graph"), ("vector", "session"), ("session", "full")]     # paired_boot(a, b) = b - a
AMEND = {}
for name, OK, keys, groups, gof in (
        ("LongMemEval-S", OK_A2, QIDS, TYPES + ["abstention"], group),
        ("LoCoMo", OK_B2, lq, list(CAT.values()), lambda x: CAT[QB[x]["category"]])):
    print(f"\n{name} (n = {len(keys)})")
    print(f"{'question group':<27}{'n':>5} | " + " | ".join(f"{m:^18}" for m in M4))
    for g in groups + ["ALL"]:
        qs = [x for x in keys if g == "ALL" or gof(x) == g]
        if not qs:
            continue
        r = {m: acc(OK, [f"{m}|{x}" for x in qs]) for m in M4}
        print(f"{g:<27}{len(qs):>5} | " + " | ".join(f"{a:5.0%} [{lo:.0%}-{hi:.0%}]".center(18) for a, lo, hi in r.values()))
    AMEND[name] = {}
    for a, b in CONTRASTS:
        dm, lo, hi = paired_boot([OK[f"{a}|{x}"] for x in keys], [OK[f"{b}|{x}"] for x in keys])
        AMEND[name][(a, b)] = (dm, lo, hi)
        print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]" + ("  resolvable" if lo > 0 or hi < 0 else "  not resolvable"))

print("\nLongMemEval:"); llm_a.print_ledger([s for s in ("nb15_sess_rerank_lme", "nb15_sess_ans_lme", "nb15_sess_judge_lme")])
print("\nLoCoMo:"); llm_b.print_ledger([s for s in ("nb15_sess_rerank_locomo", "nb15_sess_ans_locomo", "nb15_sess_judge_locomo")])
s_emb = sum(r["embed_tokens"] for r in list(SR_A.values()) + list(SR_B.values()))
print(f"Session-index embeddings: {s_emb / 1e6:.1f}M tokens, ${s_emb * 0.02 / 1e6:.2f}")


LongMemEval-S (n = 500)
question group                 n |       vector       |       graph        |        full        |      session      
single-session-user           64 |    95% [87%-98%]   |    98% [92%-100%]  |    88% [77%-94%]   |    77% [65%-85%]  
single-session-assistant      56 |    91% [81%-96%]   |    52% [39%-64%]   |   100% [94%-100%]  |    98% [91%-100%] 
single-session-preference     30 |    73% [56%-86%]   |    70% [52%-83%]   |    47% [30%-64%]   |    70% [52%-83%]  
multi-session                121 |    60% [51%-68%]   |    76% [68%-83%]   |    57% [48%-65%]   |    21% [14%-29%]  
knowledge-update              72 |    88% [78%-93%]   |    94% [87%-98%]   |    88% [78%-93%]   |    60% [48%-70%]  
temporal-reasoning           127 |    61% [53%-69%]   |    73% [65%-80%]   |    53% [44%-61%]   |    43% [35%-52%]  
abstention                    30 |    97% [83%-99%]   |    90% [74%-97%]   |    97% [83%-99%]   |    93% [79%-98%]  
ALL                          500 |    7

  full - session: +43.8% [+41.2%, +46.5%]  resolvable

LongMemEval:
stage                           calls     input tok   output tok      USD  via Batch
nb15_sess_rerank_lme              500     2,219,112       25,027     0.12       100%
nb15_sess_ans_lme                 500     1,546,970       23,874     0.08       100%
nb15_sess_judge_lme               500        96,410        2,000     0.01       100%
(embeddings)                        0             0            0     0.00         0%
TOTAL                           1,500     3,862,492       50,901     0.21

LoCoMo:
stage                           calls     input tok   output tok      USD  via Batch
nb15_sess_rerank_locomo         1,540     6,331,328       77,474     0.34       100%
nb15_sess_ans_locomo            1,540       850,469       32,879     0.05       100%
nb15_sess_judge_locomo          1,540       237,754       20,020     0.02       100%
(embeddings)                        0             0            0     0.00         0%

### Amendment results and verdict (session-level memory)

| Benchmark | Session memory | Graph − session | Session − vector | Full − session |
|---|---|---|---|---|
| LongMemEval-S (500) | 55% [51%, 60%] | +23.4 [+18.0, +28.6] | −20.0 [−24.8, −15.2] | +15.6 [+10.8, +20.2] |
| LoCoMo (1,540) | 35% [33%, 37%] | +38.3 [+35.5, +41.2] | −31.8 [−34.8, −28.8] | +43.8 [+41.2, +46.5] |

**Verdict** (sensitivity analysis; the primary result stands): the fixed session-level variant is the weakest memory
on both benchmarks, well below round-level vector memory. It does not change any conclusion: graph memory stays
ahead of vector memory, and the comparison with vector memory remains the right one.

**Why it is weak, as expected from the protocol.** The variant was fixed in advance with the same answer-context
budget as the round-level vector memory (about 2,750 tokens on LongMemEval and 420 on LoCoMo). That holds one or two
LongMemEval sessions and only part of one LoCoMo session, so most evidence is cut off: it fails worst on multi-session
(21%) and LoCoMo multi-hop (18%) questions, and does well only where one session suffices (single-session-assistant
98%). Session-level retrieval would need a larger budget to work, which the matched-budget rule rules out; the
published vector memories (about 80% on LongMemEval) use more context. We therefore do not tune it further.

Cost of the amendment: $1.67 at Batch prices ($1.06 of it for embedding the sessions).

*Note: the LongMemEval numbers of graph memory and full context in this table are the original ones, which were
affected by the session-id leak; see the next amendment for the masked numbers.*

# Amendment (2026-11-08): session ids masked in the LongMemEval prompts

*Added 2026-11-08 as a correction. The original LongMemEval numbers stay in the cells above, unchanged; the numbers
in this amendment are the corrected ones for graph memory (b) and full context (c). LoCoMo is not affected.*

**The leak.** In LongMemEval-S every evidence session id starts with `answer_` (all 500 questions), and no filler
session id does (fillers are `sharegpt_…`, `ultrachat_…` or hexadecimal ids). Two of the systems above print session ids
in the answer prompt:
- **full context (c)**: each session header is `### Session {id} ({date})`, so the answerer can see which sessions
  hold the evidence;
- **graph memory (b)**: each fact line ends `(session {id}, turn {t})` and each of the 5 source turns is headed
  `[{date} session {id} turn {t}]` (these are the only two places the graph context shows an id: 12,500 fact lines
  and 2,500 source headers over the 500 questions). The top-25 facts include an evidence session in 446 of 470
  answerable questions, so the label was visible for most of them.

Vector memory (a) and session memory (d) show dates but no ids, and LoCoMo ids are `session_N`, which carry no label;
neither is rerun.

**The fix.** Each session id is replaced by a neutral label `S1, S2, …`, numbered by date order within the question's
haystack, with the same mapping for both systems (one label per distinct id: 13 haystacks repeat a filler session on
two dates, and it keeps one label, as it kept one id). Dates, turn numbers and all other text are unchanged. The graph
is not rebuilt: extraction, entity resolution, invalidation and fact retrieval are the cached ones above, and only the
rendering of the context changes. The cell below checks, before any call, that (1) the unmasked rendering reproduces
the hash of every original cached request (1,000 of 1,000), (2) mapping the labels back gives the original context
exactly, and (3) no haystack id or evidence-id pattern is left in a masked prompt.

**Rerun.** Answer and judge steps only, with 15's own answer prompt (`ANSWER_SYS`, same user-message format,
`max_tokens=800`) and 15's own judge (LongMemEval's official type-specific prompts), all 500 questions. Reported:
masked vs original accuracy per question type (Wilson 95% intervals) and the paired-bootstrap contrasts (`paired_boot`,
5,000 resamples, seed 7) graph − full context, graph − vector memory and graph − session memory, computed with the
masked answers and shown next to the original ones.

New LLM stages (new cache entries only): `nb15_mask_ans_graph`, `nb15_mask_judge_graph`, `nb15_mask_ans_full`,
`nb15_mask_judge_full`.

In [15]:
# (e) Masking: every LongMemEval session id in the graph-memory and full-context prompts becomes a neutral label S1,
# S2, ... numbered by date order within the question's haystack (one label per distinct id, so a filler session the
# haystack repeats on two dates keeps one label, as it kept one id). Dates, turn numbers and all other text are unchanged.
# The graph is not rebuilt: the cached graph context (same extraction, resolution, invalidation and fact retrieval) is
# re-rendered by replacing the id in its only two id-bearing patterns.
FACT_ID = re.compile(r"(  \(session )(\S+)(, turn \d+\))$", re.M)                  # end of each fact line
SRC_ID = re.compile(r"^(\[\d{4}/\d\d/\d\d session )(\S+)( turn \d+\] )", re.M)       # header of each source turn
EVID_ID = re.compile(r"answer_(?:[0-9a-f]{8}|ultrachat|sharegpt)")      # shape of every evidence-session id (not e.g. `answer_start` in code)

def sess_labels(q):
    order = sorted(range(len(q["haystack_dates"])), key=lambda i: q["haystack_dates"][i])   # same order as full_context
    lab = {}
    for i in order:
        lab.setdefault(q["haystack_session_ids"][i], f"S{len(lab) + 1}")
    return lab

def render_full(q, name):
    # full_context (above) with each session id passed through `name`.
    sess = sorted(zip(q["haystack_session_ids"], q["haystack_dates"], q["haystack_sessions"]), key=lambda x: x[1])
    return "\n\n".join(f"### Session {name(sid)} ({d})\n" + "\n".join(f"{t['role'].upper()}: {t['content']}" for t in s)
                       for sid, d, s in sess)

def render_graph(ctx, name):
    sub = lambda m: m.group(1) + name(m.group(2)) + m.group(3)
    return SRC_ID.sub(sub, FACT_ID.sub(sub, ctx))

def answer_body(q, ctx):
    # The answer request of 15's answer cell, verbatim.
    return LLM.body([{"role": "system", "content": ANSWER_SYS},
                     {"role": "user", "content": f"Current date: {q['question_date']}\n\nMEMORY:\n{ctx}\n\nQuestion: {q['question']}"}],
                    max_tokens=800)

LAB = {qid: sess_labels(by_id[qid]) for qid in QIDS}
MCTX = {"graph": {}, "full": {}}
orig_rows, n_match, n_ids = llm_a.rows("answers"), 0, Counter()
for qid in QIDS:
    q, lab = by_id[qid], LAB[qid]
    inv = {v: k for k, v in lab.items()}
    g_ctx = GR[qid]["context"]
    found = [m.group(2) for m in FACT_ID.finditer(g_ctx)], [m.group(2) for m in SRC_ID.finditer(g_ctx)]
    n_ids["fact lines"] += len(found[0]); n_ids["source turns"] += len(found[1])
    assert all(s in lab for s in found[0] + found[1]), qid
    MCTX["graph"][qid] = render_graph(g_ctx, lab.__getitem__)
    MCTX["full"][qid] = render_full(q, lab.__getitem__)
    for m, orig in (("graph", g_ctx), ("full", render_full(q, lambda s: s))):
        # (1) the unmasked rendering is the request that was sent (hash of the cached original request);
        # (2) mapping the labels back gives the original context exactly, so only the ids differ;
        # (3) no session id of the haystack is left anywhere in the masked context.
        assert LLM._hash(answer_body(q, orig)) == orig_rows[f"{m}|{qid}"]["h"], (m, qid)
        back = render_graph(MCTX[m][qid], inv.__getitem__) if m == "graph" else render_full(q, lambda s: s)
        assert back == orig and render_full(q, lab.__getitem__) == MCTX["full"][qid], (m, qid)
        assert not any(s in MCTX[m][qid] for s in lab) and not EVID_ID.search(MCTX[m][qid]), (m, qid)
        n_match += 1
assert render_full(by_id[QIDS[0]], lambda s: s) == full_context(by_id[QIDS[0]])
print(f"Masking check: {n_match:,} of {2 * len(QIDS):,} unmasked prompts hash-match the cached original requests; "
      f"ids masked in the graph contexts: {n_ids['fact lines']:,} fact lines, {n_ids['source turns']:,} source-turn "
      f"headers; no session id of the haystack and no evidence-id pattern left in any masked prompt.")
ex = next((x for x in QIDS if "answer_" in GR[x]["context"]), None)
if ex:
    i = next(i for i, l in enumerate(GR[ex]["context"].splitlines()) if "(session answer_" in l)
    print("Example fact line, original vs masked:\n  ..." + GR[ex]["context"].splitlines()[i][-100:]
          + "\n  ..." + MCTX["graph"][ex].splitlines()[i][-100:])

Masking check: 1,000 of 1,000 unmasked prompts hash-match the cached original requests; ids masked in the graph contexts: 12,500 fact lines, 2,500 source-turn headers; no session id of the haystack and no evidence-id pattern left in any masked prompt.
Example fact line, original vs masked:
  ...nty, and the user said they can now focus on building a new life.  (session answer_530960c1, turn 4)
  ... of uncertainty, and the user said they can now focus on building a new life.  (session S43, turn 4)


In [16]:
# Answers and judgments with the masked contexts: 15's answer prompt and official judge, new stages only.
MASK_STAGES = {"graph": ("nb15_mask_ans_graph", "nb15_mask_judge_graph"),
               "full": ("nb15_mask_ans_full", "nb15_mask_judge_full")}
def masked_run(m):
    ans = llm_a.run(MASK_STAGES[m][0], {f"{m}|{qid}": answer_body(by_id[qid], MCTX[m][qid]) for qid in QIDS})
    jud = llm_a.run(MASK_STAGES[m][1], {k: LLM.body([{"role": "user", "content": anscheck_prompt(
        by_id[k.split("|")[1]]["question_type"], by_id[k.split("|")[1]]["question"], by_id[k.split("|")[1]]["answer"],
        a["text"], abstention=is_abs(by_id[k.split("|")[1]]))}], max_tokens=5) for k, a in ans.items()})
    return ans, jud
MANS, MOK = {}, {}
for m, (ans, jud) in zip(MASK_STAGES, pmap(masked_run, list(MASK_STAGES), workers=2)):   # the two Batch jobs run side by side
    MANS.update({k.replace(f"{m}|", f"{m}_m|"): v for k, v in ans.items()})
    MOK.update({k.replace(f"{m}|", f"{m}_m|"): "yes" in v["text"].lower() for k, v in jud.items()})
print(f"{len(MANS):,} masked answers, {len(MOK):,} judgments; answer-prompt tokens per question: "
      + ", ".join(f"{m} masked {np.mean([MANS[f'{m}_m|{q}']['in'] for q in QIDS]):,.0f} vs original {TOK_A[m]:,.0f}"
                  for m in MASK_STAGES))

1,000 masked answers, 1,000 judgments; answer-prompt tokens per question: graph masked 1,891 vs original 2,063, full masked 104,109 vs original 104,365


In [17]:
# Corrected LongMemEval results: masked vs original accuracy per question type, and the contrasts with the masked
# graph and full-context answers next to the original ones (paired_boot(a, b) = b - a; resolvable if the CI excludes 0).
OKM = dict(globals().get("OK_A2", OK_A), **MOK)
HAS_SESS = all(f"session|{q}" in OKM for q in QIDS)
COLS = ["graph", "graph_m", "full", "full_m", "vector"] + (["session"] if HAS_SESS else [])
qs_of = lambda g: [q for q in QIDS if g == "ALL" or group(q) == g]
GROUPS = [g for g in TYPES + ["abstention", "ALL"] if qs_of(g)]
print("LongMemEval-S, judge accuracy (_m = session ids masked)")
print(f"{'question group':<27}{'n':>4} | " + " | ".join(f"{m:^17}" for m in COLS))
RES_M = {}
for g in GROUPS:
    RES_M[g] = {m: acc(OKM, [f"{m}|{q}" for q in qs_of(g)]) for m in COLS}
    print(f"{g:<27}{len(qs_of(g)):>4} | " + " | ".join(f"{a:5.1%} [{lo:.0%}-{hi:.0%}]".center(17) for a, lo, hi in RES_M[g].values()))

PAIRS = [("masked - original: graph", "graph", "graph_m"), ("masked - original: full context", "full", "full_m"),
         ("graph - full context", "full", "graph"), ("graph - vector memory", "vector", "graph")]
if HAS_SESS:
    PAIRS.append(("graph - session memory", "session", "graph"))
MASKED = {"graph": "graph_m", "full": "full_m"}
DIFF_M = {}
def ci(a, b, qs):
    dm, lo, hi = paired_boot([OKM[f"{a}|{q}"] for q in qs], [OKM[f"{b}|{q}"] for q in qs])
    return dm, lo, hi
fmt = lambda d: f"{d[0]:+6.1%} [{d[1]:+.1%}, {d[2]:+.1%}]" + ("*" if d[1] > 0 or d[2] < 0 else " ")
for label, a, b in PAIRS:
    print(f"\n{label}" + ("" if label.startswith("masked") else "   (original | masked)"))
    for g in GROUPS:
        qs = qs_of(g)
        if label.startswith("masked"):
            DIFF_M[(label, g)] = ci(a, b, qs)
            print(f"  {g:<27}{fmt(DIFF_M[(label, g)])}")
        else:
            o, mk = ci(a, b, qs), ci(MASKED.get(a, a), MASKED.get(b, b), qs)
            DIFF_M[(label, g)] = {"original": o, "masked": mk}
            print(f"  {g:<27}{fmt(o)} | {fmt(mk)}")
print("\n* = 95% interval excludes 0")
print("\nCost of the masked rerun:"); llm_a.print_ledger([s for st in MASK_STAGES.values() for s in st])

LongMemEval-S, judge accuracy (_m = session ids masked)
question group                n |       graph       |      graph_m      |       full        |      full_m       |      vector       |      session     
single-session-user          64 |  98.4% [92%-100%] |  93.8% [85%-98%]  |  87.5% [77%-94%]  |  87.5% [77%-94%]  |  95.3% [87%-98%]  |  76.6% [65%-85%] 
single-session-assistant     56 |  51.8% [39%-64%]  |  48.2% [36%-61%]  | 100.0% [94%-100%] | 100.0% [94%-100%] |  91.1% [81%-96%]  |  98.2% [91%-100%]
single-session-preference    30 |  70.0% [52%-83%]  |  73.3% [56%-86%]  |  46.7% [30%-64%]  |  46.7% [30%-64%]  |  73.3% [56%-86%]  |  70.0% [52%-83%] 
multi-session               121 |  76.0% [68%-83%]  |  75.2% [67%-82%]  |  57.0% [48%-65%]  |  48.8% [40%-58%]  |  59.5% [51%-68%]  |  20.7% [14%-29%] 
knowledge-update             72 |  94.4% [87%-98%]  |  95.8% [88%-99%]  |  87.5% [78%-93%]  |  86.1% [76%-92%]  |  87.5% [78%-93%]  |  59.7% [48%-70%] 
temporal-reasoning          127 

  single-session-assistant   -46.4% [-58.9%, -33.9%]* | -50.0% [-62.5%, -37.5%]*
  single-session-preference   +0.0% [-23.3%, +23.3%]  |  +3.3% [-20.0%, +26.7%] 
  multi-session              +55.4% [+46.3%, +64.5%]* | +54.5% [+44.6%, +63.6%]*
  knowledge-update           +34.7% [+23.6%, +47.2%]* | +36.1% [+25.0%, +47.2%]*
  temporal-reasoning         +29.9% [+18.9%, +40.9%]* | +29.1% [+17.3%, +40.2%]*
  abstention                  -3.3% [-16.7%, +6.7%]  |  +3.3% [-6.7%, +13.3%] 
  ALL                        +23.4% [+18.0%, +28.6%]* | +22.8% [+17.4%, +28.2%]*

* = 95% interval excludes 0

Cost of the masked rerun:
stage                           calls     input tok   output tok      USD  via Batch
nb15_mask_ans_graph               500       945,515       22,716     0.05       100%
nb15_mask_judge_graph             500        95,270        2,000     0.01       100%
nb15_mask_ans_full                500    52,054,489       25,517     2.61       100%
nb15_mask_judge_full              500  

{'nb15_mask_ans_graph': {'calls': 500,
  'in': 945515,
  'out': 22716,
  'usd': 0.05295475000000001,
  'batch_share': 1.0},
 'nb15_mask_judge_graph': {'calls': 500,
  'in': 95270,
  'out': 2000,
  'usd': 0.005263499999999998,
  'batch_share': 1.0},
 'nb15_mask_ans_full': {'calls': 500,
  'in': 52054489,
  'out': 25517,
  'usd': 2.609103699999999,
  'batch_share': 1.0},
 'nb15_mask_judge_full': {'calls': 500,
  'in': 98047,
  'out': 2000,
  'usd': 0.005402350000000004,
  'batch_share': 1.0},
 '(embeddings)': {'calls': 0,
  'in': 0,
  'out': 0,
  'usd': 0.0,
  'batch_share': 0.0}}

### Corrected LongMemEval results (session ids masked)

| LongMemEval-S (500) | original | masked | masked − original |
|---|---|---|---|
| graph memory | 78.6% | 78.0% [74%, 81%] | −0.6 [−3.2, +1.8] |
| full context | 70.8% | 68.6% [64%, 73%] | −2.2 [−5.4, +1.0] |

| Contrast | original | masked (corrected) |
|---|---|---|
| graph − full context | +7.8 [+3.0, +12.6] | **+9.4 [+4.6, +14.0]** |
| graph − vector memory | +3.4 [−0.8, +7.4] | **+2.8 [−1.2, +7.0]** |
| graph − session memory | +23.4 [+18.0, +28.6] | **+22.8 [+17.4, +28.2]** |

**Verdict:** the leak was real but small. Masking the session ids lowers neither system resolvably overall (graph
−0.6, full context −2.2). The one resolvable change is full context on multi-session questions (−8.3 [−16.5, −0.8]):
with the ids visible, the model could find the `answer_…` sessions among about 50. Graph memory loses most on
single-session-user (−4.7, n.s.). **No conclusion changes:** graph memory stays better than full context on
LongMemEval (+9.4, slightly larger than before because full context lost more), and its difference from round-level
vector memory stays unresolvable (+2.8). Cost of the masked re-run: $2.67 at Batch prices.

*Reading:* the corrected (masked) numbers replace the original LongMemEval graph and full-context numbers wherever they
are cited; the original cells above are kept as run.